# EXP-2026-007 / Q5-D — QUALIFY (quest53)

## `EXP-2026-007 / Q5-D QUALIFY — 측정도구 자격검증`
## `NO TRAINING / NO SCIENTIFIC ANALYSIS`
## `QUALIFY RESULT NOT RUN`
## `EXP-2026-007 SCIENTIFIC RESULT NOT RUN`

**아래 셀을 실행하기 전까지 이 문서의 어떤 숫자도 결과가 아니다.**

여기서 답하는 질문은 하나뿐이다: **고정된 P파 delineation 규칙 하나가 전문가가
표시한 P파를 충분히 잘 찾는가?** 모델을 채점하지 않고, 과학적 결론도 내지 않는다.

- 선행: PREP_DATA-A `ACQUIRE_ONLY` accepted (canonical run `20260809T153151`).
- 판정 코드: `MEASUREMENT_QUALIFIED` | `MEASUREMENT_UNQUALIFIED` | `QUALIFY_RESULT_NOT_RUN`
- **`MEASUREMENT_UNQUALIFIED` 는 완전하고 유효한 결과다.** 창을 넓히거나 delineator를
  바꾸거나 더 좋아 보이는 lead로 갈아타거나 record를 손으로 빼서 되살리지 않는다.

### 여는 것 / 봉인하는 것

| | 대상 |
|---|---|
| 연다 | MIT-BIH raw `.dat`/`.hea` (channel 0), `.atr` **R 위치**, `pwave 1.0.0` 전문가 P 주석 |
| **봉인** | DS2 beat class label · V10 확률 · 처리된 beat 배열 |

`.atr` 에서 R 위치를 고르려면 "이 주석이 beat 인가"를 봐야 하므로 symbol 열을 읽긴
한다. 그래서 **읽는 즉시 boolean 으로 축약하고 class 자체는 버린다**(모듈의
`_beat_samples`). DS2 record 의 beat class 는 어떤 출력 파일에도 남지 않는다.
DS1 은 symbol 을 유지한다 — DS1 label 은 봉인 대상이 아니고, 고정 RR band 가
DS1 의 N beat 위에 정의되기 때문이다.

### record 배정 (모듈이 split 에서 계산한다)

| | records |
|---|---|
| DS1 전문가 주석 | `101 106 119 122 207 223` — dry report, 튜닝 금지 |
| DS2 전문가 주석 | `100 103 117 214 222 231` — **단 한 번** 돌리는 gate |
| DS1 전체 22개 | frozen 상수 산출 (사용자 결정 2026-08-10) |

### 실행 순서

셀 1은 mode 설정 셀이다. **freeze 가 실행 경계다** — `frozen_constants.json` 이
저장된 뒤에는 어떤 상수도 못 바꾸고, 셀 7이 저장 직전에 해시를 다시 검증한다.

1. 셀 2 — repo 준비 + commit SHA + 회귀 테스트
2. 셀 3 — Google Drive mount, 취득 자산 확인
3. 셀 4 — `DESIGN` 카드 (읽는 것 없음)
4. 셀 5 — **QUALIFY-0 환경 pin** (파형을 읽기 **전**에 저장한다)
5. 셀 6 — `QUALIFY_DS1_FREEZE` (DS1 6 dry report + DS1 22 상수 freeze)
6. 셀 7 — `QUALIFY_DS2_GATE` (**단 한 번**)
7. 셀 8 — `QUALIFY_REPORT` (저장 bundle 재표시, 재계산 없음)
8. 출력을 포함한 채로 notebook 저장 → **중단하고 보고**

### 이 단계에서 하지 않는 것 (전부 금지)

beat join · P-to-R association · S PR-AUC · SHAM permutation · 모델 학습 ·
DS2 class label 열람 · V10 확률 열람 · 파라미터 sweep · lead 교체.

**자격검증 통과는 측정도구가 쓸 만하다는 뜻이지 EXP-2026-007 의 과학적 판정이
아니다.** 다음 단계(beat join·association)는 자동 실행되지 않으며 별도 승인이 필요하다.

In [ ]:
# ── 셀 1: 실행 설정 (정확히 하나의 mode) ─────────────────────────────────────
VALID_MODES = ("DESIGN", "QUALIFY_DS1_FREEZE", "QUALIFY_DS2_GATE", "QUALIFY_REPORT")
MODE = "DESIGN"          # DESIGN -> QUALIFY_DS1_FREEZE -> QUALIFY_DS2_GATE -> QUALIFY_REPORT
assert MODE in VALID_MODES, f"MODE must be one of {VALID_MODES}"

# 병합 후에는 "main" 으로 바꾼다.
BRANCH = "claude/exp-2026-007-prep-data-intake-cb456f"
NEED_Q5DQ = 120          # 회귀 테스트 최소 통과 수 (셀 2)

print(MODE_BANNER if "MODE_BANNER" in globals() else f"mode: {MODE}")

In [ ]:
# ── 셀 2: repo 준비 + commit SHA + 회귀 테스트 ───────────────────────────────
import os, subprocess, sys

# 셀 1을 건너뛰고 여기서부터 실행해도 돌아가도록 기본값을 쓴다.
BRANCH = globals().get("BRANCH", "claude/exp-2026-007-prep-data-intake-cb456f")
NEED_Q5DQ = int(globals().get("NEED_Q5DQ", 120))
MODE = globals().get("MODE", "DESIGN")

REPO = "/content/my-github-test"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/ehdbddl06001-ui/my-github-test.git",
                    REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin", BRANCH],
               check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "-B", BRANCH,
                f"origin/{BRANCH}"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()

sys.path.insert(0, os.path.join(REPO, "mit-bih"))
for _m in ("q5d_qualify_pwave_delineator",):
    sys.modules.pop(_m, None)
import q5d_qualify_pwave_delineator as Q5DQ

_t = subprocess.run([sys.executable,
                     os.path.join(REPO, "mit-bih",
                                  "test_q5d_qualify_pwave_delineator.py")],
                    capture_output=True, text=True)
_tail = _t.stdout.strip().splitlines()[-1] if _t.stdout.strip() else "(no output)"
_n_pass = int(_tail.split("passed")[1].split("·")[0]) if "passed" in _tail else 0
assert _t.returncode == 0, f"regression tests failed:\n{_t.stdout[-3000:]}"
assert _n_pass >= NEED_Q5DQ, f"expected >= {NEED_Q5DQ} checks, got {_n_pass}"

print(Q5DQ.NO_SCIENCE_BANNER)
print(f"branch: {BRANCH} | mode: {MODE}")
print(f"commit: {COMMIT}")
print(f"module: {Q5DQ.MODULE_VERSION} {Q5DQ.MODULE_BUILD} | tests: {_tail}")
print(f"guard : {Q5DQ.assert_qualify_only()['qualify_only']}")

In [ ]:
# ── 셀 3: Google Drive mount + 취득 자산 확인 ────────────────────────────────
import os
assert "Q5DQ" in globals(), "먼저 셀 2(repo 준비)를 실행한다"
MODE = globals().get("MODE", "DESIGN")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive"
ASSET_ROOT = os.path.join(DRIVE_ROOT, Q5DQ.DRIVE_ASSET_REL)
assert os.path.isdir(ASSET_ROOT), (
    f"PREP_DATA-A 자산이 없다: {ASSET_ROOT} — 취득 단계가 먼저다")

_src = os.path.join(ASSET_ROOT, Q5DQ.SOURCE_SUBDIR)
for _sub in (Q5DQ.MITDB_DIR, Q5DQ.PWAVE_DIR):
    assert os.path.isdir(os.path.join(_src, _sub)), f"missing source/{_sub}"

print(f"asset root : {ASSET_ROOT}")
print(f"qualify dir: {Q5DQ.qualify_dir(ASSET_ROOT)}")
print(f"현재 상태   : {Q5DQ.report_bundle(ASSET_ROOT)['decision']}")

In [ ]:
# ── 셀 4: DESIGN — 규칙·gate·경계만 표시 (아무것도 읽지 않는다) ──────────────
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "DESIGN", f"이 셀은 DESIGN 전용 (지금 {MODE} — 셀 1에서 바꾼다)"

print(Q5DQ.design_card(ASSET_ROOT, MODE))
print()
print("DS1 상수 산출 범위:", len(Q5DQ.DS1_RECORDS), "records")
print("DS1 전문가 주석   :", " ".join(Q5DQ.DS1_EXPERT_RECORDS))
print("DS2 전문가 주석   :", " ".join(Q5DQ.DS2_EXPERT_RECORDS))

In [ ]:
# ── 셀 5: QUALIFY-0 환경 pin — 파형을 읽기 전에 저장한다 ─────────────────────
import json, os, time
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"

ENV_TS = time.strftime("%Y%m%dT%H%M%S")
ENV_PIN = Q5DQ.build_env_pin(ENV_TS)
_ok, _missing = Q5DQ.env_pin_is_complete(ENV_PIN)
assert _ok, f"환경 pin 불완전: {_missing} — pip install 후 다시 실행한다"

ENV_PIN_PATH = os.path.join(Q5DQ.qualify_dir(ASSET_ROOT), "runs", ENV_TS,
                            "env_pin.json")
os.makedirs(os.path.dirname(ENV_PIN_PATH), exist_ok=True)
with open(ENV_PIN_PATH, "w", encoding="utf-8") as _fh:
    json.dump(ENV_PIN, _fh, indent=2, ensure_ascii=False)

for _name, _p in ENV_PIN["packages"].items():
    print(f"  {_name:<10} {str(_p.get('version')):<10} "
          f"{str(_p.get('source_sha256'))[:32]}")
print(f"\nwaveform_read: {ENV_PIN['waveform_read']}")
print(f"saved: {ENV_PIN_PATH}")

In [ ]:
# ── 셀 6: QUALIFY_DS1_FREEZE — DS1 dry report + 상수 freeze ─────────────────
import json, os, time
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "QUALIFY_DS1_FREEZE", \
    f"이 셀은 QUALIFY_DS1_FREEZE 전용 (지금 {MODE} — 셀 1에서 바꾼 뒤 셀 1을 실행)"
assert "ENV_PIN" in globals(), "먼저 셀 5(환경 pin)를 실행한다"

TS_A = time.strftime("%Y%m%dT%H%M%S")
LOG_A = Q5DQ.RunLog()
DS1 = Q5DQ.run_ds1_freeze(ASSET_ROOT, TS_A, ENV_PIN, log=LOG_A)
FROZEN = DS1["frozen"]
Q5DQ.write_bundle(ASSET_ROOT, TS_A, MODE, FROZEN, DS1, None, LOG_A)

print()
print(json.dumps({k: FROZEN[k] for k in (
    "rr_normal_band", "discordance_threshold", "rr_band_n_normal_beats",
    "discordance_n_valid_beats", "frozen_sha256")}, indent=2))
print(f"\nfrozen -> {os.path.join(Q5DQ.qualify_dir(ASSET_ROOT), Q5DQ.FROZEN_FILE)}")
print("여기서 상수는 고정됐다. 셀 7이 저장 직전에 이 해시를 다시 검증한다.")

In [ ]:
# ── 셀 7: QUALIFY_DS2_GATE — 단 한 번만 실행한다 ─────────────────────────────
import json, os, time
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "QUALIFY_DS2_GATE", \
    f"이 셀은 QUALIFY_DS2_GATE 전용 (지금 {MODE} — 셀 1에서 바꾼 뒤 셀 1을 실행)"

_fp = os.path.join(Q5DQ.qualify_dir(ASSET_ROOT), Q5DQ.FROZEN_FILE)
assert os.path.exists(_fp), "freeze 먼저 (셀 6). 상수 없이 DS2를 열지 않는다"
with open(_fp, encoding="utf-8") as _fh:
    FROZEN = json.load(_fh)
Q5DQ.verify_frozen(FROZEN)

TS_B = time.strftime("%Y%m%dT%H%M%S")
LOG_B = Q5DQ.RunLog()
GATE = Q5DQ.run_ds2_gate(ASSET_ROOT, TS_B, FROZEN, log=LOG_B)
Q5DQ.write_bundle(ASSET_ROOT, TS_B, MODE, FROZEN, None, GATE, LOG_B)

print()
print(Q5DQ.render_gate_card(GATE["decision"], GATE["rows"]))
print(f"\nbundle : {Q5DQ.qualify_dir(ASSET_ROOT)}")
print(f"이번 실행 사본: {os.path.join(Q5DQ.qualify_dir(ASSET_ROOT), 'runs', TS_B)}")
print("다음 단계(beat join·association)는 실행되지 않았다 — 사용자 승인 대상.")

In [ ]:
# ── 셀 8: QUALIFY_REPORT — 저장 bundle만 다시 표시 (재계산 없음) ────────────
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "QUALIFY_REPORT", \
    f"이 셀은 QUALIFY_REPORT 전용 (지금 {MODE} — 셀 1에서 바꾼 뒤 셀 1을 실행)"

REP = Q5DQ.report_bundle(ASSET_ROOT)
print(f"decision   : {REP['decision']}")
print(f"recomputed : {REP['recomputed']}")
print(f"누락 파일  : {REP['missing_files']}")
print(f"보관된 실행: {REP['archived_runs']}")
print()
print(REP.get("summary", "(summary.md 없음)"))

## 마지막 화면 — 지금 상태와 다음에 할 일

- `EXP-2026-007 / Q5-D QUALIFY` · 측정도구 자격검증 · 모델 채점 없음
- 셀 7을 실행하지 않았다면 상태는 **`QUALIFY_RESULT_NOT_RUN`** 이다. 실행하지 않은
  notebook 을 `MEASUREMENT_QUALIFIED` 로 표시하지 않는다.
- 저장 위치: `MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data/qualify/`
  · 최신 bundle 은 `qualify/`, 불변 보관본은 `qualify/runs/<timestamp>/`
  · **canonical evidence 는 notebook 출력이 아니라 이 bundle 이다.**
- 실패했다면 `decision.json` 의 `first_stopping_reason` 과
  `pwave_qualification.csv` 의 해당 record 행을 그대로 보고한다. 임계값을 내리거나
  record 를 빼서 되살리지 않는다.

### 자격검증에 실패하면

`MEASUREMENT_UNQUALIFIED` 는 **완전하고 유효한 결과**다 — "지금 이 도구로는 잴 수
없다". 이 경우 EXP-2026-007 의 과학 실험은 여기서 종료되고, beat join 과
association 은 실행하지 않는다.

### 자격검증을 통과해도

통과는 **측정도구가 쓸 만하다**는 뜻일 뿐이다. EXP-2026-007 의 과학적 판정은 여전히
**NOT RUN** 이고 spec status 는 `approved_for_implementation` 그대로다.
다음 단계인 beat join 은 별도 설계 검토와 **별도 승인**을 거친 뒤에만 시작한다.
알려진 위험: `.atr` sample ↔ 처리 beat identity 조인은 Q5-A 실측에서 성공률
**1.9%(우연 수준)** 였다. 이 문제는 Codex 가 병렬로 설계 중이다.

### 여기서 멈춘다

beat join · P-to-R 계산 · DS2 outcome 분석 · S PR-AUC · SHAM permutation · 모델
학습은 이 notebook 에 **경로 자체가 없다**(모듈의 `FORBIDDEN_TOKENS` 가 소스 수준에서
막는다). 다음 substage 는 이 자격검증 bundle 을 사람이 검토하고 별도로 승인한
뒤에만 시작한다.